In [ ]:
# =============================
# 1. Setup & Imports
# =============================
!pip install torch torchvision kaggle matplotlib scikit-learn

import os
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms
from sklearn.model_selection import train_test_split
import numpy as np
from PIL import Image
import matplotlib.pyplot as plt

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", device)

# =============================
# 2. Download small dataset
# (Amazon Rainforest subset)
# =============================
os.environ['KAGGLE_USERNAME'] = "your_kaggle_username"
os.environ['KAGGLE_KEY'] = "your_kaggle_api_key"

!kaggle datasets download -d nikitarom/planets-dataset -p ./data
!unzip -q ./data/planets-dataset.zip -d ./data/rainforest

# For demo: use a very small subset
image_dir = "./data/rainforest/planet/planet/train-jpg"
label_dir = "./data/rainforest/planet/planet/train_v2.csv"

# =============================
# 3. Dummy Dataset for Segmentation
# (Turn multi-label CSV into simple Forest vs Non-Forest masks)
# =============================

class DummyForestDataset(Dataset):
    def __init__(self, img_dir, transform=None, n_samples=200):
        self.img_dir = img_dir
        self.images = [os.path.join(img_dir, f) for f in os.listdir(img_dir)[:n_samples]]
        self.transform = transform

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):
        img = Image.open(self.images[idx]).convert("RGB").resize((128,128))
        # Generate dummy binary mask: threshold green channel
        mask = np.array(img)[:,:,1] > 100
        mask = Image.fromarray(mask.astype(np.uint8)*255).resize((128,128))

        if self.transform:
            img = self.transform(img)
            mask = self.transform(mask)

        mask = (mask > 0).float()
        return img, mask

transform = transforms.Compose([
    transforms.ToTensor()
])

dataset = DummyForestDataset(image_dir, transform)
train_size = int(0.8*len(dataset))
val_size = len(dataset)-train_size
train_ds, val_ds = torch.utils.data.random_split(dataset, [train_size,val_size])

train_loader = DataLoader(train_ds, batch_size=8, shuffle=True)
val_loader = DataLoader(val_ds, batch_size=8)

# =============================
# 4. Define U-Net
# =============================
class DoubleConv(nn.Module):
    def __init__(self, in_c, out_c):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(in_c,out_c,3,padding=1), nn.ReLU(),
            nn.Conv2d(out_c,out_c,3,padding=1), nn.ReLU()
        )
    def forward(self,x): return self.conv(x)

class UNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.d1 = DoubleConv(3,64)
        self.d2 = DoubleConv(64,128)
        self.d3 = DoubleConv(128,256)
        self.u2 = DoubleConv(256+128,128)
        self.u1 = DoubleConv(128+64,64)
        self.out = nn.Conv2d(64,1,1)

        self.pool = nn.MaxPool2d(2)
        self.up = nn.Upsample(scale_factor=2,mode="bilinear",align_corners=True)

    def forward(self,x):
        c1 = self.d1(x)
        c2 = self.d2(self.pool(c1))
        c3 = self.d3(self.pool(c2))
        u2 = self.up(c3)
        u2 = self.u2(torch.cat([u2,c2],dim=1))
        u1 = self.up(u2)
        u1 = self.u1(torch.cat([u1,c1],dim=1))
        return torch.sigmoid(self.out(u1))

model = UNet().to(device)
criterion = nn.BCELoss()
optimizer = optim.Adam(model.parameters(), lr=1e-3)

# =============================
# 5. Training Loop
# =============================
epochs = 3
for epoch in range(epochs):
    model.train()
    loss_sum = 0
    for imgs, masks in train_loader:
        imgs, masks = imgs.to(device), masks.to(device)
        optimizer.zero_grad()
        preds = model(imgs)
        loss = criterion(preds, masks)
        loss.backward()
        optimizer.step()
        loss_sum += loss.item()
    print(f"Epoch {epoch+1}, Loss: {loss_sum/len(train_loader):.4f}")

# =============================
# 6. Save model
# =============================
os.makedirs("./models", exist_ok=True)
torch.save(model.state_dict(), "./models/unet_model.pth")
print("✅ Model saved as unet_model.pth")
